In [1]:
%load_ext autoreload
%autoreload 2
import torch
from src.data.preprocessor import read_smard_data
from src.pipeline.pipeline import ForecastingPipeline
from src.predictors.chronos import Chronos
from pathlib import Path
import pandas as pd
import numpy as np

/home/skowronek/.conda/envs/mt_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/skowronek/.conda/envs/mt_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
2025-06-18 07:57:57,243 - INFO - config.py - PyTorch version 2.5.1 available.


In [2]:
data, mapping, freq = read_smard_data(
    file_paths=[
        "data/electricity_consumption/Actual_consumption_201501010000_202001010000_Quarterhour.csv",
        "data/electricity_consumption/Actual_consumption_202001010000_202506120000_Quarterhour.csv",
    ],
    selected_time_series=["grid load [MWh] Original resolutions", "Residual load [MWh] Original resolutions"],
    freq=pd.Timedelta("15 min"),
)

2025-06-17 13:41:27,245 - INFO - preprocessor.py - Reading SMARD data...
2025-06-17 13:41:27,246 - INFO - preprocessor.py - Reading file: data/electricity_consumption/Actual_consumption_201501010000_202001010000_Quarterhour.csv
2025-06-17 13:41:28,298 - INFO - preprocessor.py - Reading file: data/electricity_consumption/Actual_consumption_202001010000_202506120000_Quarterhour.csv
2025-06-17 13:41:29,422 - INFO - preprocessor.py - Total rows after concat: 366236
2025-06-17 13:41:29,427 - INFO - preprocessor.py - Filtering columns: ['grid load [MWh] Original resolutions', 'Residual load [MWh] Original resolutions']
2025-06-17 13:41:29,430 - INFO - preprocessor.py - Columns retained: ['Start date', 'grid load [MWh] Original resolutions', 'Residual load [MWh] Original resolutions']
2025-06-17 13:41:29,479 - INFO - preprocessor.py - Reshaped DataFrame: 732392 rows
2025-06-17 13:41:29,520 - INFO - preprocessor.py - Mapped 2 unique time series.
2025-06-17 13:41:29,522 - INFO - preprocessor.py

In [3]:
model_name = "chronos-bolt-tiny"
lead_times = np.arange(1, 192 + 1).tolist()
quantiles = np.round(np.arange(0.1, 1, 0.1), 1).tolist()
test_start_date = pd.Timestamp("2023-01-01")
val_window_size = pd.DateOffset(years=1)

if torch.cuda.is_available():
    device_map = "cuda"
elif torch.mps.is_available():
    device_map = "mps"
else:
    device_map = "cpu"

In [5]:
output_dir = Path("./results/warm-up-ratio/electricity-consumption-with-warmup/pipeline/")

# chronos zero shot results
pipeline = ForecastingPipeline(
    model=Chronos,
    model_kwargs={
        "pretrained_model_name_or_path": f"amazon/{model_name}",
        "device_map": device_map,
        "lead_times": lead_times,
        "freq": freq,
        "finetuning_type": "full",
        "finetuning_hp_search": False,
    },
    postprocessors=None,
    postprocessor_kwargs=None,
    output_dir=output_dir / f"{model_name}-finetuned-full",
)

results = pipeline.backtest(
    data=data,
    test_start_date=test_start_date,
    rolling_window_eval=False,
    train=True,
    val_window_size=val_window_size,
    train_window_size=None,
    test_window_size=None,
    calibration_based_on="val",
    save_results=True,
)



2025-06-17 13:14:37,800 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-tiny
2025-06-17 13:14:37,801 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-tiny
2025-06-17 13:14:38,476 - INFO - pipeline.py - Start E2E backtesting...
2025-06-17 13:14:38,481 - INFO - pipeline.py - Starting data split operation.
2025-06-17 13:14:38,510 - INFO - pipeline.py - Data split operation completed successfully.
2025-06-17 13:14:38,511 - INFO - pipeline.py - evaluate is set to true. Removing all item_ids which contain only nans in target column.
2025-06-17 13:14:38,525 - INFO - pipeline.py - Starting training process from 2015-01-01 00:00:00 to 2021-12-31 23:45:00
2025-06-17 13:14:38,529 - INFO - pipeline.py - Validation data from 2022-01-01 00:00:00 to 2022-12-31 23:45:00
2025-06-17 13:14:38,529 - INFO - pipeline.py - Initializing predictor with model: Chronos
2025-06-17 13:14:38,530 - INFO - chronos.py - Loading Chronos pipeline from mod

{'eval_loss': 16.39311981201172, 'eval_runtime': 29.8142, 'eval_samples_per_second': 2350.292, 'eval_steps_per_second': 73.455, 'epoch': 0}


Could not estimate the number of tokens of the input, floating-point operations will not be computed


{'loss': 15.0201, 'grad_norm': 184.69908142089844, 'learning_rate': 1.6663045839669781e-06, 'epoch': 0.05006192555895965}
{'eval_loss': 13.861680030822754, 'eval_runtime': 29.4034, 'eval_samples_per_second': 2383.126, 'eval_steps_per_second': 74.481, 'epoch': 0.1000586663190144}
{'loss': 14.1554, 'grad_norm': 159.1455841064453, 'learning_rate': 3.334781664132088e-06, 'epoch': 0.1001238511179193}
{'loss': 13.8318, 'grad_norm': 180.49928283691406, 'learning_rate': 5.003258744297198e-06, 'epoch': 0.15018577667687896}
{'eval_loss': 13.243579864501953, 'eval_runtime': 29.4016, 'eval_samples_per_second': 2383.275, 'eval_steps_per_second': 74.486, 'epoch': 0.2001173326380288}
{'loss': 13.112, 'grad_norm': 189.21981811523438, 'learning_rate': 6.6717358244623076e-06, 'epoch': 0.2002477022358386}
{'loss': 12.8394, 'grad_norm': 224.12925720214844, 'learning_rate': 8.340212904627416e-06, 'epoch': 0.25030962779479826}
{'eval_loss': 12.974554061889648, 'eval_runtime': 29.4773, 'eval_samples_per_seco

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].
2025-06-17 13:33:17,424 - INFO - chronos.py - Saved fine-tuned model to results/warm-up-ratio/electricity-consumption-with-warmup/pipeline/chronos-bolt-tiny-finetuned-full/models/finetuned-full/fine-tuned-ckpt.
2025-06-17 13:33:17,426 - INFO - chronos.py - Loading Chronos pipeline from model: results/warm-up-ratio/electricity-consumption-with-warmup/pipeline/chronos-bolt-tiny-finetuned-full/models/finetuned-full/fine-tuned-ckpt
2025-06-17 13:33:17,426 - INFO - chronos.py - Initializing Chronos pipeline with model: results/warm-up-ratio/electricity-consumption-with-warmup/pipeline/chronos-bolt-tiny-finetuned-full/models/finetuned-full/fine-tuned-ckpt


{'eval_loss': 12.523544311523438, 'eval_runtime': 29.0962, 'eval_samples_per_second': 2408.289, 'eval_steps_per_second': 75.268, 'epoch': 1.000586663190144}
{'train_runtime': 1116.7234, 'train_samples_per_second': 1318.737, 'train_steps_per_second': 41.213, 'train_loss': 12.205097083672639, 'epoch': 1.000586663190144}


2025-06-17 13:33:17,478 - INFO - chronos.py - Trained model has been automatically loaded from checkpoint.
2025-06-17 13:33:17,478 - INFO - base.py - Time to fit Chronos in seconds: 1118.43
2025-06-17 13:33:17,479 - INFO - pipeline.py - Pipeline training completed in 1118.958898 seconds.
2025-06-17 13:33:17,520 - INFO - pipeline.py - Starting prediction for test data from 2023-01-01 00:00:00 to 2025-06-11 23:45:00
2025-06-17 13:33:17,521 - INFO - pipeline.py - Running prediction using the model: Chronos
Predicting using Chronos:   0%|          | 0/10971 [00:00<?, ?it/s]/home/skowronek/master-thesis/chronos-forecasting/src/chronos/chronos_bolt.py:627: UserWarning: We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
  warnings.warn(msg)
Predicting using Chronos: 100%|██████████| 10971/10971 [04:32<00:00, 40.28it/s]
2025-06-17 13:37:56,461 - INFO - pipeline.py - Prediction completed successfully.
2025-06-17 

In [4]:
output_dir = Path("./results/no-warm-up-ratio/electricity-consumption-with-warmup/pipeline/")

# chronos zero shot results
pipeline = ForecastingPipeline(
    model=Chronos,
    model_kwargs={
        "pretrained_model_name_or_path": f"amazon/{model_name}",
        "device_map": device_map,
        "lead_times": lead_times,
        "freq": freq,
        "finetuning_type": "full",
        "finetuning_hp_search": False,
    },
    postprocessors=None,
    postprocessor_kwargs=None,
    output_dir=output_dir / f"{model_name}-finetuned-full",
)

results = pipeline.backtest(
    data=data,
    test_start_date=test_start_date,
    rolling_window_eval=False,
    train=True,
    val_window_size=val_window_size,
    train_window_size=None,
    test_window_size=None,
    calibration_based_on="val",
    save_results=True,
)


2025-06-17 13:41:36,617 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-tiny
2025-06-17 13:41:36,618 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-tiny
2025-06-17 13:41:37,252 - INFO - pipeline.py - Start E2E backtesting...
2025-06-17 13:41:37,258 - INFO - pipeline.py - Starting data split operation.
2025-06-17 13:41:37,303 - INFO - pipeline.py - Data split operation completed successfully.
2025-06-17 13:41:37,304 - INFO - pipeline.py - evaluate is set to true. Removing all item_ids which contain only nans in target column.
2025-06-17 13:41:37,318 - INFO - pipeline.py - Starting training process from 2015-01-01 00:00:00 to 2021-12-31 23:45:00
2025-06-17 13:41:37,320 - INFO - pipeline.py - Validation data from 2022-01-01 00:00:00 to 2022-12-31 23:45:00
2025-06-17 13:41:37,321 - INFO - pipeline.py - Initializing predictor with model: Chronos
2025-06-17 13:41:37,321 - INFO - chronos.py - Loading Chronos pipeline from mod

{'eval_loss': 16.39311981201172, 'eval_runtime': 29.4271, 'eval_samples_per_second': 2381.207, 'eval_steps_per_second': 74.421, 'epoch': 0}


Could not estimate the number of tokens of the input, floating-point operations will not be computed


{'loss': 13.887, 'grad_norm': 186.0897216796875, 'learning_rate': 9.833344197466485e-06, 'epoch': 0.05006192555895965}
{'eval_loss': 13.170100212097168, 'eval_runtime': 29.8692, 'eval_samples_per_second': 2345.96, 'eval_steps_per_second': 73.32, 'epoch': 0.1000586663190144}
{'loss': 13.0955, 'grad_norm': 136.95062255859375, 'learning_rate': 9.666471112269954e-06, 'epoch': 0.1001238511179193}
{'loss': 12.8873, 'grad_norm': 175.7505645751953, 'learning_rate': 9.49959802707342e-06, 'epoch': 0.15018577667687896}
{'eval_loss': 12.820581436157227, 'eval_runtime': 30.0359, 'eval_samples_per_second': 2332.944, 'eval_steps_per_second': 72.913, 'epoch': 0.2001173326380288}
{'loss': 12.318, 'grad_norm': 228.33538818359375, 'learning_rate': 9.332724941876889e-06, 'epoch': 0.2002477022358386}
{'loss': 12.1573, 'grad_norm': 218.15692138671875, 'learning_rate': 9.165851856680357e-06, 'epoch': 0.25030962779479826}
{'eval_loss': 12.668766975402832, 'eval_runtime': 29.6754, 'eval_samples_per_second': 23

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].
2025-06-17 14:00:16,219 - INFO - chronos.py - Saved fine-tuned model to results/no-warm-up-ratio/electricity-consumption-with-warmup/pipeline/chronos-bolt-tiny-finetuned-full/models/finetuned-full/fine-tuned-ckpt.
2025-06-17 14:00:16,220 - INFO - chronos.py - Loading Chronos pipeline from model: results/no-warm-up-ratio/electricity-consumption-with-warmup/pipeline/chronos-bolt-tiny-finetuned-full/models/finetuned-full/fine-tuned-ckpt
2025-06-17 14:00:16,222 - INFO - chronos.py - Initializing Chronos pipeline with model: results/no-warm-up-ratio/electricity-consumption-with-warmup/pipeline/chronos-bolt-tiny-finetuned-full/models/finetuned-full/fine-tuned-ckpt


{'eval_loss': 12.489848136901855, 'eval_runtime': 30.3193, 'eval_samples_per_second': 2311.136, 'eval_steps_per_second': 72.231, 'epoch': 1.000586663190144}
{'train_runtime': 1116.775, 'train_samples_per_second': 1318.676, 'train_steps_per_second': 41.211, 'train_loss': 11.784014085403095, 'epoch': 1.000586663190144}


2025-06-17 14:00:16,294 - INFO - chronos.py - Trained model has been automatically loaded from checkpoint.
2025-06-17 14:00:16,295 - INFO - base.py - Time to fit Chronos in seconds: 1118.45
2025-06-17 14:00:16,295 - INFO - pipeline.py - Pipeline training completed in 1118.982035 seconds.
2025-06-17 14:00:16,318 - INFO - pipeline.py - Starting prediction for test data from 2023-01-01 00:00:00 to 2025-06-11 23:45:00
2025-06-17 14:00:16,319 - INFO - pipeline.py - Running prediction using the model: Chronos
Predicting using Chronos:   0%|          | 0/10971 [00:00<?, ?it/s]/home/skowronek/master-thesis/chronos-forecasting/src/chronos/chronos_bolt.py:627: UserWarning: We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
  warnings.warn(msg)
Predicting using Chronos: 100%|██████████| 10971/10971 [04:35<00:00, 39.85it/s]
2025-06-17 14:04:58,265 - INFO - pipeline.py - Prediction completed successfully.
2025-06-17 

In [ ]:
# get evaluation
from src.core.timeseries_evaluation import load_predictions, get_crps_scores

In [4]:
predictions = load_predictions(prediction_dirs = ["results/no-warm-up-ratio", "results/warm-up-ratio"])

2025-06-18 07:58:42,909 - INFO - timeseries_evaluation.py - Loading predictions by searching in provided directories...
2025-06-18 07:58:42,911 - INFO - timeseries_evaluation.py - Common path identified: results


2025-06-18 07:58:43,685 - INFO - timeseries_evaluation.py - Loaded prediction file: `results/no-warm-up-ratio/electricity-consumption-with-warmup/pipeline/chronos-bolt-tiny-finetuned-full/backtest/Chronos/predictions.joblib` as key: no-warm-up-ratio_electricity-consumption-with-warmup_pipeline_chronos-bolt-tiny-finetuned-full_Chronos
2025-06-18 07:58:44,450 - INFO - timeseries_evaluation.py - Loaded prediction file: `results/warm-up-ratio/electricity-consumption-with-warmup/pipeline/chronos-bolt-tiny-finetuned-full/backtest/Chronos/predictions.joblib` as key: warm-up-ratio_electricity-consumption-with-warmup_pipeline_chronos-bolt-tiny-finetuned-full_Chronos
2025-06-18 07:58:44,451 - INFO - timeseries_evaluation.py - Finished loading predictions. 
 
  Loaded keys:
      - no-warm-up-ratio_electricity-consumption-with-warmup_pipeline_chronos-bolt-tiny-finetuned-full_Chronos
      - warm-up-ratio_electricity-consumption-with-warmup_pipeline_chronos-bolt-tiny-finetuned-full_Chronos


In [11]:
get_crps_scores(predictions)

Compute CRPS score: 100%|██████████| 2/2 [00:28<00:00, 14.46s/it]


,warm-up-ratio_electricity-consumption-with-warmup_pipeline_chronos-bolt-tiny-finetuned-full_Chronos,no-warm-up-ratio_electricity-consumption-with-warmup_pipeline_chronos-bolt-tiny-finetuned-full_Chronos
lead times,,
1,135.326862,129.891417
2,146.773320,141.322183
3,159.802399,155.762786
4,175.025389,171.560296
5,196.627831,193.532804
...,...,...
189,1079.379540,1083.049781
190,1080.502796,1083.372522
191,1080.382247,1083.005363
